# Choosing and Running a Test

This notebook is the program described in Section 5 of *Probability and Statistics: A Concise Guide*. It does two things:

1. It asks how the data were collected and what is being tested, then **names** the test that those answers commit you to.
2. It **runs** that test, using SciPy and statsmodels, and writes a report in the language of Section 4.

You do not write any statistical code. Run the cells in order. The questionnaire cell displays dropdowns; the later cells replay examples you have already seen in Section 4.

## How to start

Open this file from Jupyter Notebook or JupyterLab, using your **Anaconda** kernel. Then run the next cell once, so Python can find the `test_chooser` package sitting in this same folder.


In [ ]:
from pathlib import Path
import sys

code_dir = Path.cwd()
if not (code_dir / "test_chooser").is_dir():
    raise SystemExit(
        "Set the notebook's working directory to the Prob_and_Stat folder "
        "(the folder that contains test_chooser) and re-run this cell."
    )

sys.path.insert(0, str(code_dir))

from test_chooser import (
    StudySpec,
    analyze,
    show_chooser,
    sample_from_summary,
    sample_from_values,
    load_file,
    battery_report,
    ping_pong_report,
    material_report,
    proportion_report,
    write_example_files,
)

write_example_files()
print("Ready. Example files are in:", code_dir / "examples")


## The questions the program asks

The program does not look at a file and guess the test. That would hide the very decision Section 5 is meant to make visible. It asks:

- What is being tested: one mean, two means, one proportion, or two proportions.
- For two means: are the samples paired, or independent?
- For one mean: is the population standard deviation known? If it is unknown but $n > 30$, do you want the large-sample $z$ approximation used in Section 4.5 and in the Type II example of Section 4.6?
- For two independent means: do you assume equal variances? (Yes: the pooled $t$-test of the battery example. No: Welch's $t$-test.)
- The alternative hypothesis: two-sided, upper-tailed, or lower-tailed.
- The hypothesized value and the level of significance $\alpha$.
- Whether the data come from a designed experiment or an observational study. That answer does not change the arithmetic. It changes the last sentence of the report.

Then you supply the data as a summary ($n$, mean, $s$, or successes), as a pasted list, or as a CSV/Excel file, one sample per column.

The report names the test, states the assumptions that choice has just committed you to, shows how those assumptions can be checked with the tools of Section 4.2, plugs the numbers into the formula, and gives both a critical value and a $p$-value, as Section 4.6 does. It also gives a two-sided confidence interval and recalls the agreement between that interval and a two-sided test (Section 4.6.8).


## Questionnaire

Run the next cell. Fill in the answers, then click **Choose and run the test**.


In [ ]:
show_chooser()

## Example 1. Battery lifetimes (Section 4.6)

The maker of Battery Type X wants to claim that those batteries last, on average, 3 or more months longer than Type Y. The hypotheses are

$$
H_0:\ \mu_X - \mu_Y \le 3, \qquad H_1:\ \mu_X - \mu_Y > 3,
$$

with $\alpha = .05$. The two samples are independent, $\sigma$ is unknown, and equal variances are assumed, so the program should choose the **pooled two-sample $t$-test** and reproduce Table 22.

The data are in `examples/batteries.csv`. The next cell also runs the same analysis from the values stored in the package.


In [ ]:
from test_chooser.io_data import load_file
from test_chooser.examples import battery_spec

print(battery_report())

print("\n--- same analysis from the CSV file ---\n")
cols = load_file(code_dir / "examples" / "batteries.csv")
print(analyze(battery_spec(), cols[0], cols[1]))


## Example 2. Ping-pong balls (Section 4.6)

Twenty samples of 100 balls are drawn from a vat. Each entry is the number of blue balls in a sample of 100. The claim to be tested is that the mean count is 50 (so the vat is half blue). The population standard deviation is unknown and $n = 20$ is not large, so the program should choose a **one-sample $t$-test**.

The second run uses $\mu_0 = 47$, which was chosen after looking at the sample mean. Section 4.6 already warns that this is an illustration of the mechanics and not a real test. The program will still do the arithmetic. It cannot know that $H_0$ was revised after the data were seen; that warning has to be supplied by the reader, as it is here.


In [ ]:
print(ping_pong_report(50))
print("\n--- H0 revised to 47 after seeing the data (not a real test) ---\n")
print(ping_pong_report(47))


## Example 3. Material strength (Section 4.6, Type II example)

Fifty samples have mean 25.9 and sample standard deviation 4.3. The manufacturer wants a mean strength of at least 25:

$$
H_0:\ \mu \le 25, \qquad H_1:\ \mu > 25, \qquad \alpha = .05.
$$

The book uses the **large-sample $z$ approximation** because $n = 50 > 30$, even though $\sigma$ is estimated by $s$. Check the box in the questionnaire that requests that approximation, or run the first command below. The second command is the one-sample $t$-test on the same summary, which is the default when $\sigma$ is unknown.

The book’s $z$-calculator reported $z = 1.48026$ and a one-sided $p$-value of about $.0694$. Both routes fail to reject $H_0$.


In [ ]:
print(material_report(use_z=True))
print("\n--- same summaries, but with the t-test ---\n")
print(material_report(use_z=False))


## Example 4. The vat as a proportion

The ping-pong data can also be read another way. The first sample in Table 21 is a single draw of 100 balls with 47 blues, a sample proportion $\tilde p = .47$. Testing $H_0:\ p = .5$ is then a **one-sample $z$-test for a proportion**, not a $t$-test for a mean of twenty counts.

A mean count of 50 in samples of 100 is the same hypothesis as $p = .5$. What differs is the amount of data: the first run below uses one sample of 100 balls, while the $t$-test used all twenty. The second run uses all twenty samples, 938 blue balls out of 2000.

In [ ]:
print(proportion_report())
print("\n--- all twenty samples: 938 blue balls out of 2000 ---\n")
print(proportion_report(all_samples=True))

## How to read the report

- **Test chosen** is the decision the questionnaire has just made. If it is not the test you intended, change an answer; the program will not silently switch tests.
- **Assumptions** are not optional extras. They are what the named test requires. Skewness near 0 and kurtosis near 3, as in the battery write-up of Section 4.6, support normality; they do not prove it. Pairing and independence are taken from your answers, not tested.
- **Arithmetic** shows the same formula as the book, with the numbers filled in. SciPy or statsmodels supply the $p$-value and the critical value.
- **Conclusion** uses the language of Section 4.6: reject or do not reject. Failing to reject is not proof of $H_0$.
- **Matching confidence interval** is the two-sided interval for the same parameter. For a two-sided test it agrees with the test about the hypothesized value (Section 4.6.8).
- **Caveats** repeat the distinction, from the end of Section 4.9, between a designed experiment and observational data.

Regression, Latin-square ANOVA, and nonparametric tests are not in this first version.
